# Universidad Nacional de Colombia  
## Sede Bogotá  
### Facultad de Ingeniería
### Departamento de Ingeniería de Sistemas e Industrial

---

## **Inteligencia Artificial y Mini-robots**

**Título del Taller:**  
*Algoritmos Genéticos*

**Autor:**  
*Juan Jeronimo Gómez Rubiano*  

**Periodo:**  
*2025-2*

**Fecha de entrega:**  
*Septiembre 18/2025*

---

*El source code junto a este documento es encuentran disponible en el siguiente repositorio público de github: https://github.com/jujgomezru/ia-minirobots*

## Ejercicio 1

### Maximizar la función f(x) = x sen(10 π x) + 1, con x ∈ [0,1].

Podemos utilizar un algoritmo genético simple para maximizar la función dada.

Para comenzar, vamos a importar las librerías necesarias:

In [1]:
import math               # Libreria que se usa para realizar operaciones matemáticas y funciones
import random             # Libreria usada para generar números aleatorios
import numpy as np        # Encuentra una raiz real entre 0.5 y 1.5

Primero, definimos la nueva ecuación, así como la longitud del cromosoma, para asegurar que los cromosomas individuales sean computables por Jupyter:

In [2]:
# Parámetros del algoritmo genético
longCrom = 20
K = 50  # Tamaño de la población
M = 100  # Número de generaciones
pm = 0.05  # Probabilidad de mutación

# Nueva función a maximizar: f(x) = x*sen(10πx) + 1
def ecuacion(x):
    valor = x * math.sin(10 * math.pi * x) + 1
    return valor

A continuación, definimos la primera generación de cromosomas de manera aleatoria. En la función decodificar, definimos los límites superior e inferior del intervalo a evaluar en la función. También se adecúa el valor binario literal del cromosoma, como el valor de x correspondiente para la función a estudiar.
Una vez se define cómo se decodifica el cromosoma, se evalúa el valor real del cromosoma en la función matemática, con el método res_Funcion. Luego, con el método evalua, se evalúa la aptitud de cada cromosoma, y se obtienen los porcentajes de aptitud de la población, los cuales usaremos más adelante para la ruleta de selección. Se realiza un proceso artificial para asegurarse de que no hayan aptitudes negativas, como consecuencia de que es posible obtener números positivos y negativos, pero no pueden existir aptitudes negativas si estas representan probabilidades.

In [3]:
def genera(K, longCrom):
    Pob_nueva = np.zeros([K, longCrom], dtype=int)
    for j in range(K):
        cromosoma = [random.randint(0, 1) for _ in range(longCrom)]
        Pob_nueva[j] = cromosoma
    return Pob_nueva

def decodificar(cromosoma):
    xi = 0.0
    xf = 1.0
    Max = 2 ** longCrom
    cromPot = [cromosoma[i] * 2 ** (longCrom - i - 1) for i in range(longCrom)]
    valorDecimal = sum(cromPot)
    valDeco = xi + (xf - xi) * valorDecimal / Max
    return valDeco

def res_Funcion(cromosoma):
    x = decodificar(cromosoma)
    funcion = ecuacion(x)
    return funcion, x

def evalua(Pob_nueva):
    vectorX = np.zeros(K, dtype=float)
    aptitud = np.zeros(K, dtype=float)
    
    for i in range(K):
        t, x = res_Funcion(Pob_nueva[i])
        vectorX[i] = x
        aptitud[i] = t
    min_apt = min(aptitud)
    if min_apt < 0:
        aptitud = aptitud - min_apt + 0.001
    
    Apt_total = float(sum(aptitud))
    probab = [j/Apt_total for j in aptitud]
    
    return probab, vectorX, aptitud

Primero, se define el método de la selección por ruleta. Para hacer la selección, se define un número aleatorio, y luego se va recorriendo el porcentaje acumulado de aptitud entre todos los cromosomas. Cuando se supera el número aleatorio, se escoge el cromosoma para la siguiente generación. De esta manera, todos los cromosomas tienen posibilidad de pasar a la siguiente generación, pero la probabilidad de ser escogido es mayor para los cromosomas con mayor aptitud.

Adicionalmente, se definen los métodos de cruce y mutación. Para realizar el cruce, se usa una copia de la población seleccionada, y se escogen de manera aleatoria la probabilidad, y la longitud de la porción cruzada. Los cromosomas se intercambian en la porción distal de cruzamiento, y son añadidos a la nueva población. Por otra parte, para la mutación, se seleccionan los cromosomas, y sus respectivos bits, al azar, y se cambia su contenido cuando son elegidos.

In [4]:
def seleccion_ruleta(poblacion, probabilidad):
    chosen = []
    for _ in range(K):
        r = random.random()
        acumulado = 0
        for i in range(len(poblacion)):
            acumulado += probabilidad[i]
            if r <= acumulado:
                chosen.append(poblacion[i].copy())
                break
    return np.array(chosen)

def cruce(Pob_nueva, prob_cromosoma):
    nueva_poblacion = Pob_nueva.copy()
    maxIndex = np.argmax(prob_cromosoma)
    
    for i in range(0, K-1, 2):
        if random.random() < 0.8:
            rand = random.randint(1, longCrom-1)
            padre1 = Pob_nueva[i].copy()
            padre2 = Pob_nueva[i+1].copy()
            
            # Realizar cruce
            hijo1 = np.concatenate([padre1[:rand], padre2[rand:]])
            hijo2 = np.concatenate([padre2[:rand], padre1[rand:]])
            
            nueva_poblacion[i] = hijo1
            nueva_poblacion[i+1] = hijo2
    
    nueva_poblacion[0] = Pob_nueva[maxIndex]
    return nueva_poblacion

def muta(Pob_nueva, pm):
    poblacion_mutada = Pob_nueva.copy()
    for i in range(K):
        for j in range(longCrom):
            if random.random() < pm:
                poblacion_mutada[i][j] = 1 - poblacion_mutada[i][j]
    return poblacion_mutada



Ahora que se han definido las funciones, se aplica el algoritmo genético para resolver el problema de optimización planteado. Primero, se genera una población de 50 cromosomas de 20 bits. Se inician las variables globales "mejor_global" en menos infinito, y "mejor_x_global" en 0 para garantizar que la mejor solución provenga de la población creada.

A continuación, se evalúan todos los cromosomas, se identifica el cromosoma con mejor aptitud (junto con el valor de x correspondiente), y una vez se definen las aptitudes individuales, se realiza la selección, cruce y mutación, tal como se definió previamente. Cada ciclo de esta rutina corresponde a una generación de la población, y se ejecuta el ciclo por 100 generaciones. Cada generación va a imprimir el valor de x con mayor aptitud, y al final se retorna el mejor valor de x. 

In [6]:
Pob_nueva = genera(K, longCrom)
mejor_global = -float('inf')
mejor_x_global = 0

for i in range(M):
    prob_cromosoma, vectorX, aptitudes = evalua(Pob_nueva)
    mejor_idx = np.argmax(aptitudes)
    mejor_aptitud = aptitudes[mejor_idx]
    mejor_x = vectorX[mejor_idx]
    if mejor_aptitud > mejor_global:
        mejor_global = mejor_aptitud
        mejor_x_global = mejor_x
    
    print(f"Generación {i}: Mejor x = {mejor_x:.6f}, Mejor valor = {mejor_aptitud:.6f}")
    Pob_seleccionada = seleccion_ruleta(Pob_nueva, prob_cromosoma)
    Pob_cruzada = cruce(Pob_seleccionada, prob_cromosoma)
    Pob_nueva = muta(Pob_cruzada, pm)

print("\n" + "="*50)
print(f"Mejor solución encontrada: x = {mejor_x_global:.6f}")
print(f"Valor de la función: f(x) = {mejor_global:.6f}")
print("="*50)

Generación 0: Mejor x = 0.851629, Mejor valor = 1.850514
Generación 1: Mejor x = 0.656335, Mejor valor = 1.643380
Generación 2: Mejor x = 0.637078, Mejor valor = 1.585302
Generación 3: Mejor x = 0.637204, Mejor valor = 1.586408
Generación 4: Mejor x = 0.869739, Mejor valor = 1.707808
Generación 5: Mejor x = 0.869739, Mejor valor = 1.707808
Generación 6: Mejor x = 0.869738, Mejor valor = 1.707822
Generación 7: Mejor x = 0.838487, Mejor valor = 1.784233
Generación 8: Mejor x = 0.836045, Mejor valor = 1.756982
Generación 9: Mejor x = 0.835973, Mejor valor = 1.756106
Generación 10: Mejor x = 0.835973, Mejor valor = 1.756106
Generación 11: Mejor x = 0.854066, Mejor valor = 1.847108
Generación 12: Mejor x = 0.846255, Mejor valor = 1.840406
Generación 13: Mejor x = 0.846255, Mejor valor = 1.840406
Generación 14: Mejor x = 0.846259, Mejor valor = 1.840422
Generación 15: Mejor x = 0.855837, Mejor valor = 1.841488
Generación 16: Mejor x = 0.858708, Mejor valor = 1.826772
Generación 17: Mejor x =

A continuación, se muestra la gráfica de la función a resolver, y se compara la solución encontrada:

Como se puede observar, el software gráfico GeoGebra muestra como punto más alto del dominio como x = , lo cual es en efecto una aproximación muy cercana al valor presentado por el algoritmo genético.

## Ejercicio 2

### Verdadera democracia. Suponga que usted es el jefe de gobierno y está interesado en que pasen los proyectos de su programa político. Sin embargo, en el congreso conformado por 5 partidos, no es fácil su tránsito, por lo que debe repartir el poder, conformado por ministerios y otras agencias del gobierno, con base en la representación de cada partido. Cada entidad estatal tiene un peso de poder, que es el que se debe distribuir. Suponga que hay 50 curules, distribuya aleatoriamente, con una distribución no informe entre los 5 partidos esas curules. Defina una lista de 50 entidades y asígneles aleatoriamente un peso político de 1 a 100 puntos. Cree una matriz de poder para repartir ese poder, usando AGs.

In [11]:
import numpy as np
from numpy.random import randint, rand
import random

El objetivo de este programa, es encontrar, en una distribución aleatoria no uniforme de tamaño de 5 partidos, distribuir a los 50 políticos en distintas curules, con distintos pesos políticos, para hacer la distribución entre partidos lo más uniforme posible. Primero, vamos a definir las funciones de generación de partidos y curules de manera aleatoria. Usamos para este fin la distribución de dirichlet:

In [12]:
def generar_curules():
    """Genera una distribución no uniforme de 50 curules entre 5 partidos"""
    proporciones = np.random.dirichlet(np.ones(5))
    curules = (proporciones * 50).round().astype(int)
    diferencia = 50 - sum(curules)
    if diferencia != 0:
        idx = np.argmax(curules)
        curules[idx] += diferencia
    return curules.tolist()

def generar_pesos_entidades():
    """Genera 50 pesos aleatorios entre 1 y 100 para las entidades"""
    return np.random.randint(1, 101, 50).tolist()

def generar_individuo(curules):
    """Genera un individuo que respeta exactamente la distribución de curules"""
    individuo = []
    for partido, count in enumerate(curules):
        individuo.extend([partido] * count)
    individuo = np.array(individuo)
    np.random.shuffle(individuo)
    return individuo.tolist()

Con la base para crear partidos y curules de manera aleatoria, adaptamos las funciones del Ejercicio 1, para definir los pasos del algoritmo genético. Para hacer esto, debemos identificar qué distingue al ejercicio actual del anterior. Sobre todo a la hora de calcular la aptitud. Para este ejemplo, un cromosoma consiste en una distribución especial de curules. Para definir su aptitud, se calcula el peso de poder de cada partido en el cromosoma, y se calcula la desviación estándar entre los partidos. Se aplica una función inversa para asignar mayor aptitud a los cromosomas on menor desviación estándar. La lógica detrás del cruce y la mutación es la misma, pero los cromosomas son significativamente más complejos, porque cada uno de los 50 genes tiene 5 opciones, en vez de una, y deben realizarse comprobaciones de que el cromosoma hijo conserve el número original de políticos de cada partido, y solo cambie la asignación de los políticos.

In [13]:
def calcular_aptitud(individuo, pesos_entidades):
    """Calcula la aptitud basada en la equidad de la distribución de poder"""
    pesos_partidos = [0] * 5
    for i, partido in enumerate(individuo):
        pesos_partidos[partido] += pesos_entidades[i]
    
    # Calculamos la desviación estándar de los pesos por partido
    desviacion = np.std(pesos_partidos)
    
    # Convertimos en aptitud (menor desviación -> mayor aptitud)
    return 1 / (1 + desviacion)

def cruzar(padre1, padre2, curules):
    """Cruce que mantiene exactamente la distribución de curules"""
    # Creamos un hijo como copia del padre1
    hijo = padre1.copy()
    
    # Identificamos las posiciones donde los padres difieren
    diferencias = [i for i in range(len(padre1)) if padre1[i] != padre2[i]]
    
    # Intercambiamos valores entre el hijo y el padre2 para mejorar la diversidad
    for i in diferencias:
        if np.random.random() < 0.5:  # 50% de probabilidad de intercambiar
            hijo[i] = padre2[i]
    
    # Verificamos y corregimos para mantener la distribución exacta
    conteo = [0] * 5
    for partido in hijo:
        conteo[partido] += 1
    
    # Aseguramos que se mantenga la distribución exacta
    for partido in range(5):
        while conteo[partido] > curules[partido]:
            # Buscamos un partido que tenga menos de lo debido
            for partido2 in range(5):
                if conteo[partido2] < curules[partido2]:
                    # Encontramos una posición para cambiar
                    for i in range(len(hijo)):
                        if hijo[i] == partido:
                            hijo[i] = partido2
                            conteo[partido] -= 1
                            conteo[partido2] += 1
                            break
                    break
    
    return hijo

def mutar(individuo, curules):
    """Mutación que mantiene la distribución de curules"""
    # Intercambiamos dos asignaciones aleatorias
    i, j = np.random.choice(len(individuo), 2, replace=False)
    individuo[i], individuo[j] = individuo[j], individuo[i]
    
    # Verificamos y corregimos la distribución si es necesario
    conteo = [0] * 5
    for partido in individuo:
        conteo[partido] += 1
    
    # Aseguramos que se mantenga la distribución exacta
    for partido in range(5):
        while conteo[partido] != curules[partido]:
            if conteo[partido] > curules[partido]:
                # Buscamos un partido que tenga menos
                for partido2 in range(5):
                    if conteo[partido2] < curules[partido2]:
                        # Cambiamos una asignación de este partido al otro
                        for k in range(len(individuo)):
                            if individuo[k] == partido:
                                individuo[k] = partido2
                                conteo[partido] -= 1
                                conteo[partido2] += 1
                                break
                        break
            else:
                # Buscamos un partido que tenga más
                for partido2 in range(5):
                    if conteo[partido2] > curules[partido2]:
                        # Cambiamos una asignación de otro partido a este
                        for k in range(len(individuo)):
                            if individuo[k] == partido2:
                                individuo[k] = partido
                                conteo[partido] += 1
                                conteo[partido2] -= 1
                                break
                        break
    
    return individuo



Ahora se procede a diseñar el algoritmo genético como tal. Como se comentó previamente, el principal aspecto a variar es la forma del "cromosoma". En vez de crear un número binario que represente un valor contenido por un intervalo, toca construirlo como una distribución dada de curules, la cual no se representa de manera directa como un número binario. 

In [14]:
def algoritmo_genetico_distribucion_poder(M, K, curules, pesos_entidades, tasa_mutacion=0.1):
    """Algoritmo genético para distribuir el poder entre partidos"""
    poblacion = [generar_individuo(curules) for _ in range(K)]
    mejor_aptitud = -1
    mejor_individuo = None
    
    for generacion in range(M):
        # Evaluar aptitud
        aptitudes = [calcular_aptitud(ind, pesos_entidades) for ind in poblacion]
        max_aptitud = max(aptitudes)
        
        if max_aptitud > mejor_aptitud:
            mejor_aptitud = max_aptitud
            mejor_individuo = poblacion[np.argmax(aptitudes)].copy()
        
        # Selección por ruleta
        total_aptitud = sum(aptitudes)
        if total_aptitud == 0:
            probabilidades = [1/len(aptitudes)] * len(aptitudes)
        else:
            probabilidades = [apt / total_aptitud for apt in aptitudes]
        
        # Crear nueva población
        nueva_poblacion = []
        for _ in range(K):
            # Seleccionar padres
            idx_padre1 = np.random.choice(len(poblacion), p=probabilidades)
            idx_padre2 = np.random.choice(len(poblacion), p=probabilidades)
            
            padre1 = poblacion[idx_padre1].copy()
            padre2 = poblacion[idx_padre2].copy()
            
            # Cruzar
            hijo = cruzar(padre1, padre2, curules)
            
            # Mutar
            if np.random.random() < tasa_mutacion:
                hijo = mutar(hijo, curules)
            
            nueva_poblacion.append(hijo)
        
        poblacion = nueva_poblacion
    
    return mejor_individuo

Finalmente, realizamos una ejecución del algoritmo para determinar qué tan eficiente es para garantizar una distribución equitativa entre los partidos. Para el análisis que se presenta a continuación se usa la semilla 42, una población de 100 cromosomas, y se aplica por 1000 generaciones. Para hacer más sencilla la visualización, se omite la visualización de la evolución de la aptitud, pero si se imprime la distribución resultante, mostrando como el algoritmo intenta clasificar a las curules, de manera que los 5 partidos tengan el poder más equitativo posible.

In [15]:
# Generar datos iniciales
np.random.seed(42)
curules = generar_curules()
pesos_entidades = generar_pesos_entidades()

# Ejecutar el algoritmo genético
mejor_asignacion = algoritmo_genetico_distribucion_poder(
    M=1000, 
    K=100, 
    curules=curules, 
    pesos_entidades=pesos_entidades,
    tasa_mutacion=0.1
)

# Calcular resultados finales
pesos_partidos = [0] * 5
for i, partido in enumerate(mejor_asignacion):
    pesos_partidos[partido] += pesos_entidades[i]

peso_total = sum(pesos_entidades)
objetivo = peso_total * 0.2

# Imprimir resultados
print("DISTRIBUCIÓN DE CURULES POR PARTIDO:")
print("-" * 40)
for i, count in enumerate(curules):
    print(f"Partido {i+1}: {count} curules")

print(f"\nPESO TOTAL DE TODAS LAS ENTIDADES: {peso_total}")
print(f"OBJETIVO IDEAL POR PARTIDO: {objetivo:.2f} (20% del total)")

print("\nDISTRIBUCIÓN FINAL DE PODER:")
print("-" * 40)
for i in range(5):
    diferencia = pesos_partidos[i] - objetivo
    porcentaje_diferencia = (diferencia / objetivo) * 100
    print(f"Partido {i+1}: {pesos_partidos[i]:6.1f} pesos "
          f"(Diferencia: {diferencia:6.1f}, {porcentaje_diferencia:6.1f}%)")

print("\nDETALLE DE ASIGNACIÓN POR CURUL:")
print("-" * 40)
print("Curul  Peso  Partido")
print("-" * 40)
for i in range(50):
    print(f"{i+1:5d}  {pesos_entidades[i]:4d}  {mejor_asignacion[i]+1:7d}")

print("\nESTADÍSTICAS FINALES:")
print("-" * 40)
diferencias = [abs(p - objetivo) for p in pesos_partidos]
print(f"Diferencia promedio: {sum(diferencias)/5:.2f}")
print(f"Diferencia máxima: {max(diferencias):.2f}")
print(f"Desviación estándar: {np.std(pesos_partidos):.2f}")

DISTRIBUCIÓN DE CURULES POR PARTIDO:
----------------------------------------
Partido 1: 4 curules
Partido 2: 26 curules
Partido 3: 11 curules
Partido 4: 8 curules
Partido 5: 1 curules

PESO TOTAL DE TODAS LAS ENTIDADES: 2472
OBJETIVO IDEAL POR PARTIDO: 494.40 (20% del total)

DISTRIBUCIÓN FINAL DE PODER:
----------------------------------------
Partido 1:  340.0 pesos (Diferencia: -154.4,  -31.2%)
Partido 2:  722.0 pesos (Diferencia:  227.6,   46.0%)
Partido 3:  684.0 pesos (Diferencia:  189.6,   38.3%)
Partido 4:  626.0 pesos (Diferencia:  131.6,   26.6%)
Partido 5:  100.0 pesos (Diferencia: -394.4,  -79.8%)

DETALLE DE ASIGNACIÓN POR CURUL:
----------------------------------------
Curul  Peso  Partido
----------------------------------------
    1    83        1
    2    87        4
    3    75        4
    4    75        3
    5    88        4
    6   100        5
    7    24        2
    8     3        2
    9    22        2
   10    53        2
   11     2        2
   12    88   

En el ejercicio de la semilla 42, la distribución por partidos es: [4,26,11,8,1]. Por lo tanto, es evidente que, a menos de que la diferencia de poder entre curules sea muy diferente entre ellas, va a ser muy difícil realizar una distribución equitativa. En los resultados finales se observa que el partido unipersonal se le asigna la curul 6, la cual tiene un poder de 100 (máximo), y todas las curules con poco poder son asignadas al partido mayoritario (Partido 2). 

## Ejercicio 3

### Una empresa proveedora de energía eléctrica dispone de cuatro plantas de generación para satisfacer la demanda diaria de energía eléctrica en Cali, Bogotá, Medellín y Barranquilla. Cada una puede generar 3, 6, 5 y 4 GW al día respectivamente. Las necesidades de Cali, Bogotá, Medellín y Barranquilla son de 4, 3, 5 y 3 GW al día respectivamente. Los costos por el transporte de energía por cada GW entre plantas y ciudades se dan en la siguiente tabla:

|          | Cali | Bogotá | Medellín | Barranq. |
|----------|:----:|:------:|:--------:|:--------:|
| Planta C |   1  |    4   |     3    |     6    |
| Planta B |   4  |    1   |     4    |     5    |
| Planta M |   3  |    4   |     1    |     4    |
| Planta A|   6  |    5   |     4    |     1    |

### Los costos del KW-H por generador se dan en la siguiente tabla:

| Generador | $KW-H |
|-----------|:-----:|
| Planta C  |  680  |
| Planta B  |  720  |
| Planta M  |  660  |
| Planta A |  750  |

### Encontrar usando AGs el mejor despacho de energía minimizando los costos de transporte y generación.

In [7]:
!pip install deap

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.9/135.9 kB 1.4 MB/s eta 0:00:0000:01


In [21]:
import random
import numpy as np
from deap import base, creator, tools

Primero, vamos a definir los valores de entrada como variables, arreglos y matrices, así como los parámetros de ejecución del algoritmo genético. En este ejemplo, a diferencia de los anteriores, definimos un factor de penalización para cromosomas que incumplan las directrices dadas. Esto es por que, queremos que los cromosomas incorrectos tengan menor chance de ser seleccionados, pero no queremos descartarlos por completo, en caso de que tengan características que puedan generar hijos más aptos por el cruce con otro cromosoma.

In [23]:
# Datos del problema (versión original que funcionaba)
cost_list = [
    17.32, 20.32, 19.32, 22.32,   # Planta C
    21.28, 18.28, 21.28, 22.28,   # Planta B
    18.84, 19.84, 16.84, 19.84,   # Planta M
    24.00, 23.00, 22.00, 19.00    # Planta A
]

# Capacidades de las plantas (en GW-día)
plant_cap = [3, 6, 5, 4]

# Demandas de las ciudades (en GW-día)
city_demand = [4, 3, 5, 3]

# Límites superiores para cada variable x_ij (en GW-día)
bounds = [
    3, 3, 3, 3,   # Planta C
    4, 3, 5, 3,   # Planta B
    4, 3, 5, 3,   # Planta M
    4, 3, 4, 3    # Planta A
]

# Parámetros del algoritmo genético
POPULATION_SIZE = 300
P_CROSSOVER = 0.5
P_MUTATION = 0.2
MAX_GENERATIONS = 2500
PENALTY_FACTOR = 1000  # Factor de penalización para restricciones

Para este problema usamos Deap, que es una librería en Python para computación evolutiva. Deap tiene múltiples clases que realizan las mismas tareas aplicadas en los ejercicios anteriores, pero de maneras optimizadas y ampliamente aceptadas por la comunidad.
En este ejemplo, primero se crean dos tipos, el primer tipo se llama "FitnessMin" y hereda el optimizador de puntos mínimos por defecto de la librería. Esto permite usar múltiples métodos predefinidos El segundo tipo se llama "Individual" y va a almacenar los valores de aptitud de cada cromosoma en la generación.
Para usar las funcionalidades de Deap, se inicializa un Toolbox. Este es un contenedor, en donde se pueden ejecutar métodos de computación evolutiva, así como registrar y organizar operadores genéticos. Por ejemplo, al inicio, con el método "register" definimos tres funciones: La función "attr_values" define la manera de crear valores aleatorios según los límites dados por el problema. La función "individual" crea cromosomas, y la función "population" crea una población de cromosomas.

In [24]:
# Crear tipos de fitness e individuo
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Inicializar toolbox
toolbox = base.Toolbox()

# Función para generar un individuo aleatorio
def random_value(bounds):
    return [random.randint(0, bound) for bound in bounds]

# Registrar la función de inicialización
toolbox.register("attr_values", random_value, bounds)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.attr_values)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

Ahora, definimos la función de evaluación. Para esta, calculamos el costo total de general la cantidad de energía definida por el cromosoma. Una vez se define el costo de esta implementación, se calcula el flujo por planta y por ciudad, para determinar si viola alguna de las restricciones. Se aplica entonces, una penalización a la aptitud por flujo de planta superior al límite, y por flujo de ciudad inferior al límite. De esta manera se asegura que la solución de "no gastar nada", sea penalizada porque no resuelve el problema, y las soluciones más ambiciosas son penalizadas por superar las capacidades de producción de todas las plantas.

Finalmente, se implementan en el toolbox las funciones de evaluación, cruce, mutación y selección, sin tener que definir todos los detalles de su lógica interna, como se hizo en los ejercicios anteriores.

In [25]:
# Función de evaluación
def evaluate(individual):
    # Calcular el costo total de transporte y generación
    total_cost = sum(cost * value for cost, value in zip(cost_list, individual))
    
    # Calcular el flujo total por planta
    plant_flow = [
        sum(individual[0:4]),   # Planta C
        sum(individual[4:8]),   # Planta B
        sum(individual[8:12]),  # Planta M
        sum(individual[12:16])  # Planta A
    ]
    
    # Calcular el flujo total por ciudad
    city_flow = [
        individual[0] + individual[4] + individual[8] + individual[12],   # Cali
        individual[1] + individual[5] + individual[9] + individual[13],   # Bogotá
        individual[2] + individual[6] + individual[10] + individual[14],  # Medellín
        individual[3] + individual[7] + individual[11] + individual[15]   # Barranquilla
    ]
    
    # Inicializar penalización
    penalty = 0
    
    # Penalizar por exceso de capacidad en plantas
    for flow, cap in zip(plant_flow, plant_cap):
        if flow > cap:
            penalty += PENALTY_FACTOR * (flow - cap)
    
    # Penalizar por desviación de la demanda en ciudades
    for flow, demand in zip(city_flow, city_demand):
        if flow != demand:
            penalty += PENALTY_FACTOR * abs(flow - demand)
    
    # Fitness es el costo total más la penalización
    return total_cost + penalty,

# Registrar operadores genéticos
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=0, up=bounds, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

Finalmente, se ejecuta el algoritmo. La ejecución en este caso es notoriamente más sencilla, ya que las rutinas fueron definidas previamente con la ayuda de Deap. Primero se construye una población, y se evalúan sus aptitudes globales. Luego, se itera el ciclo de evolución a tavés de las generaciones definiendo la selección, cruce y mutación. También se incluye una rutina sencilla de manejo de errores para individuos con aptitud inválida.

In [27]:
# Función principal para ejecutar el algoritmo genético
def main():
    # Inicializar población
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Evaluar toda la población
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
        
    # Estadísticas
    fits = [ind.fitness.values[0] for ind in population]
    g = 0
    
    # Evolución
    while g < MAX_GENERATIONS:
        g += 1
        
        # Selección de la siguiente generación
        offspring = toolbox.select(population, len(population))
        offspring = list(map(toolbox.clone, offspring))
        
        # Aplicar crossover y mutación
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < P_CROSSOVER:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values
                
        for mutant in offspring:
            if random.random() < P_MUTATION:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                
        # Evaluar individuos con fitness inválido
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
            
        # Reemplazar población
        population[:] = offspring
        
        # Recopilar estadísticas y mostrar progreso cada 100 generaciones
        fits = [ind.fitness.values[0] for ind in population]
        
        if g % 50 == 0:
            best_index = np.argmin(fits)
            best_fit = fits[best_index]
            print(f"Generación {g}: Mejor fitness = {best_fit}")
        
    # Encontrar el mejor individuo
    best_index = np.argmin(fits)
    best = population[best_index]
    return best

# Ejecutar el algoritmo genético
print("Iniciando algoritmo genético...")
best_solution = main()

# Imprimir la solución en formato matricial
print("\nMejor solución encontrada (matriz de asignación en GW-día):")
print("          Cali  Bogotá  Medellín  Barranq.")
plant_names = ["Planta C", "Planta B", "Planta M", "Planta A"]
for i in range(4):
    row = best_solution[i*4:(i+1)*4]
    print(f"{plant_names[i]:<10} {row[0]:<6} {row[1]:<7} {row[2]:<9} {row[3]:<8}")

# Calcular el flujo por planta y ciudad para la solución
plant_flow = [
    sum(best_solution[0:4]),
    sum(best_solution[4:8]),
    sum(best_solution[8:12]),
    sum(best_solution[12:16])
]
city_flow = [
    best_solution[0] + best_solution[4] + best_solution[8] + best_solution[12],
    best_solution[1] + best_solution[5] + best_solution[9] + best_solution[13],
    best_solution[2] + best_solution[6] + best_solution[10] + best_solution[14],
    best_solution[3] + best_solution[7] + best_solution[11] + best_solution[15]
]

print("\nFlujo por planta:", plant_flow, "(Capacidad:", plant_cap, ")")
print("Flujo por ciudad:", city_flow, "(Demanda:", city_demand, ")")

# Calcular el costo sin penalizaciones para ver el costo real
real_cost = sum(cost * value for cost, value in zip(cost_list, best_solution))
print("Costo total (sin penalizaciones):", real_cost, "millones de $")

Iniciando algoritmo genético...
Generación 50: Mejor fitness = 281.72
Generación 100: Mejor fitness = 280.88
Generación 150: Mejor fitness = 280.88
Generación 200: Mejor fitness = 280.88
Generación 250: Mejor fitness = 271.28000000000003
Generación 300: Mejor fitness = 271.28000000000003
Generación 350: Mejor fitness = 271.28000000000003
Generación 400: Mejor fitness = 271.28000000000003
Generación 450: Mejor fitness = 271.28000000000003
Generación 500: Mejor fitness = 271.28000000000003
Generación 550: Mejor fitness = 271.28000000000003
Generación 600: Mejor fitness = 271.28000000000003
Generación 650: Mejor fitness = 271.28000000000003
Generación 700: Mejor fitness = 271.28000000000003
Generación 750: Mejor fitness = 271.28000000000003
Generación 800: Mejor fitness = 271.28000000000003
Generación 850: Mejor fitness = 271.28000000000003
Generación 900: Mejor fitness = 271.28000000000003
Generación 950: Mejor fitness = 271.28000000000003
Generación 1000: Mejor fitness = 271.28000000000

Como se observa en el ejemplo simulado, la solución más optima consiste en:
Planta C suministra todos sus 3 GW a Cali.
Planta B suministra 3 GW a Bogotá, y 1 a Medellín.
Planta M suministra 4 GW a Medellín, y 1 a Cali.
Planta A suministra 3 GW a Barranquilla.
Esta solución cumple con el límite mínimo de demanda, consiguiendo el menor costo posible, en este caso $271.28 millones de pesos.